# 16차시 실습 — 내 서비스를 자동으로 테스트하기

> **day1 연속 실습.** 15차시에서 **평가**하던 그 기능(`my_service`)을, 오늘은 **매번 자동으로 지키는 테스트**로 굳힙니다.
> **테스트 자동 생성 → 회귀(스냅샷) 비교 → AI 디버깅** 을 한 흐름으로 직접 돌립니다.

## 🧪 사용법
- 워크샵 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**해 보는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다.
- 위에서부터 **순서대로** 실행하세요 (`Shift+Enter`).

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오 — 내 서비스를 자동으로 테스트

15차시: 내 서비스(`my_service`)가 **'그럴듯하게'가 아니라 '정확하게'** 동작하는지 **평가**했습니다(한 번 보는 것).
16차시: 그 기준을 **자동으로 돌아가는 테스트**로 굳힙니다 — 코드를 바꿀 때마다 **매번 자동으로** 확인.

> 공통 예시 = **맛집 추천 서비스**(day1·day2에서 쓰던 그 데이터). 구조는 동일하니 **팀 MVP의 기능**으로 바꾸면 그대로 적용됩니다(STEP 8).

## STEP 1 — 내 서비스 정의 (`my_service`)

AI 기능은 보통 **두 부분**으로 나뉩니다.
1. **결정적 로직** (예: 예산 필터) → **단위 테스트**로 자동 검증 (STEP 2~3)
2. **LLM 출력** (예: 한 줄 추천 카피) → **스냅샷 회귀**로 변화 감지 (STEP 4)

⌨️ **터미널 Codex:** `codex "맛집 추천 서비스의 예산 필터 로직과 한 줄 추천 카피 기능을 만들어줘"`

In [2]:
# day1·day2에서 쓰던 맛집 데이터 (실제로는 내 앱의 DB/API 자리)
RESTAURANTS = [
    {"name": "매운갈비찜집", "price": 18000, "taste": "매움"},
    {"name": "순한국밥",     "price": 9000,  "taste": "순함"},
    {"name": "불막창",       "price": 22000, "taste": "매움"},
    {"name": "치즈파스타",   "price": 15000, "taste": "순함"},
]

def within_budget(restaurants, budget):
    """예산(budget) 이하 맛집만 골라 가격 오름차순으로 반환 (서비스의 결정적 핵심 로직).

    - budget < 0 이면 ValueError.
    - 예산 내 후보가 없으면 빈 리스트.
    """
    if budget < 0:
        raise ValueError("budget은 0 이상이어야 합니다")
    picked = [r for r in restaurants if r["price"] <= budget]
    return sorted(picked, key=lambda r: r["price"])

def my_service(request, budget=20000):
    """사용자 요청을 받아 예산 내 맛집을 추천하고 LLM이 한 줄 카피를 붙이는 AI 기능."""
    cands = within_budget(RESTAURANTS, budget)
    if not cands:
        return "예산 내 추천 맛집이 없어요."
    top = cands[0]
    r = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[
            {"role": "system", "content": "맛집을 한국어 한 문장으로 매력적으로 추천하라. 한 문장만."},
            {"role": "user", "content": f"요청:{request} / 추천 대상:{top['name']}({top['price']}원)"},
        ],
    )
    return r.choices[0].message.content.strip()

# 함수 소스를 문자열로 확보 (모델에게 통째로 넘기기 위해)
import inspect
TARGET_SRC = inspect.getsource(within_budget)
print(TARGET_SRC)
print("빠른 확인:", my_service("강남에서 싼 점심", budget=12000))

def within_budget(restaurants, budget):
    """예산(budget) 이하 맛집만 골라 가격 오름차순으로 반환 (서비스의 결정적 핵심 로직).

    - budget < 0 이면 ValueError.
    - 예산 내 후보가 없으면 빈 리스트.
    """
    if budget < 0:
        raise ValueError("budget은 0 이상이어야 합니다")
    picked = [r for r in restaurants if r["price"] <= budget]
    return sorted(picked, key=lambda r: r["price"])

빠른 확인: 강남에서 9,000원에 맛볼 수 있는 순한국밥은 신선한 재료와 깊은 국물 맛으로 점심을 풍성하게 채워줄 최고의 선택입니다!


## STEP 2 — 테스트를 자동 생성 (코드 → 테스트 위임)

서비스의 **결정적 로직**(`within_budget`) 소스를 통째로 모델에 주고 **assert 단위 테스트**를 만들게 합니다.
좋은 프롬프트의 3요소 — **① 시그니처 ② 정상 경로 ③ 엣지·예외** — 를 명시하는 게 핵심입니다.

⌨️ **터미널 Codex:** `codex "within_budget 함수의 정상·경계·예외 케이스를 assert 단위 테스트로 만들어줘"`

In [3]:
GEN_PROMPT = f"""다음 파이썬 함수에 대한 단위 테스트를 작성하라.

{TARGET_SRC}

요구사항:
- 각 테스트는 `def test_...():` 형태의 함수로, 내부에서 `assert` 사용.
- 정상 경로(예산 내 후보가 가격 오름차순으로 반환), 경계값(딱 맞는 가격, 빈 결과), 예외(budget<0 → ValueError)를 모두 포함.
- 예외는 try/except로 ValueError가 나는지 확인.
- 함수 본문은 다시 정의하지 말고, `within_budget`가 이미 있다고 가정.
- 테스트용 입력 데이터는 테스트 안에서 직접 만들어 사용.
- 코드만 출력. 설명·마크다운 펜스 없이.
"""

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": GEN_PROMPT}],
    temperature=0,
)
generated = resp.choices[0].message.content
generated = generated.replace("```python", "").replace("```", "").strip()
print(generated)

def test_within_budget_normal_case():
    restaurants = [
        {"name": "Restaurant A", "price": 10},
        {"name": "Restaurant B", "price": 20},
        {"name": "Restaurant C", "price": 15}
    ]
    budget = 20
    result = within_budget(restaurants, budget)
    assert result == [
        {"name": "Restaurant A", "price": 10},
        {"name": "Restaurant C", "price": 15},
        {"name": "Restaurant B", "price": 20}
    ]

def test_within_budget_boundary_case():
    restaurants = [
        {"name": "Restaurant A", "price": 10},
        {"name": "Restaurant B", "price": 20}
    ]
    budget = 20
    result = within_budget(restaurants, budget)
    assert result == [
        {"name": "Restaurant A", "price": 10},
        {"name": "Restaurant B", "price": 20}
    ]

def test_within_budget_empty_result():
    restaurants = [
        {"name": "Restaurant A", "price": 30},
        {"name": "Restaurant B", "price": 40}
    ]
    budget = 20
    result = within_budget(restaurants, bu

## STEP 3 — 생성된 테스트를 미니 러너로 실행

생성 코드를 실행해 `test_`로 시작하는 함수를 모아 하나씩 돌리고 **pass/fail**을 셉니다.
> ⚠️ 생성 테스트는 **사람이 한 번 읽고** 의도와 맞는지 확인하세요. AI는 그럴듯한 **틀린 기대값**도 씁니다.

In [4]:
def run_tests(test_source, target_globals):
    """test_source(문자열)를 실행해 test_* 함수를 모아 돌리고 결과 출력."""
    ns = dict(target_globals)          # 대상 함수가 보이도록 같은 네임스페이스에
    exec(test_source, ns)
    tests = {k: v for k, v in ns.items() if k.startswith("test_") and callable(v)}
    passed = failed = 0
    for name, fn in tests.items():
        try:
            fn()
            print(f"✅ PASS · {name}")
            passed += 1
        except Exception as e:
            print(f"❌ FAIL · {name} → {type(e).__name__}: {e}")
            failed += 1
    print(f"\n결과: {passed} passed, {failed} failed ({len(tests)} tests)")
    return passed, failed

run_tests(generated, {"within_budget": within_budget})

✅ PASS · test_within_budget_normal_case
✅ PASS · test_within_budget_boundary_case
✅ PASS · test_within_budget_empty_result
✅ PASS · test_within_budget_negative_budget

결과: 4 passed, 0 failed (4 tests)


(4, 0)

## STEP 4 — 회귀(Regression) 데모: 출력 스냅샷 → 변경 → diff

`my_service`의 **LLM 출력**은 비결정적이라 회귀를 놓치기 쉽습니다. 한 번 합격한 출력을 **스냅샷**으로 저장해 두고,
요청/예산을 바꾼 뒤 **diff**로 변화를 잡아냅니다. (재현성을 위해 `my_service`는 `temperature=0`)

⌨️ **터미널 Codex:** `codex "my_service의 출력을 스냅샷으로 저장하고, 변경 후 diff를 보여줘"`

In [5]:
import difflib

REQUEST, BUDGET = "강남에서 매운 점심 추천해줘", 20000
snapshot = my_service(REQUEST, BUDGET)         # ① 합격한 출력을 스냅샷으로 저장
print("📸 스냅샷 저장:\n", snapshot)

# ② 입력 조건을 바꾼다 (회귀를 일으킬 수 있는 변경: 예산 축소)
new_output = my_service(REQUEST, budget=10000)

# ③ diff — 달라졌으면 사람이 판단 (의도된 변화면 스냅샷 갱신, 아니면 회귀)
diff = list(difflib.unified_diff(
    snapshot.splitlines(), new_output.splitlines(),
    fromfile="snapshot", tofile="new", lineterm=""))
if diff:
    print("\n⚠️ 출력 변화 감지 (diff) — 개선인가 회귀인가?")
    print("\n".join(diff))
else:
    print("\n✅ 스냅샷과 동일 — 회귀 없음")

📸 스냅샷 저장:
 강남의 순한국밥에서 매콤한 국물과 함께 푸짐한 한 끼를 즐기며, 진정한 한국의 맛을 느껴보세요!

✅ 스냅샷과 동일 — 회귀 없음


## STEP 5 — AI 기반 디버깅: 버그 주입 → 테스트가 잡고 → AI가 고친다

내 서비스에 **일부러 버그**(예산 필터 누락)를 심어, ① STEP 2의 테스트가 **회귀를 잡는지** 확인하고
② 실패한 테스트의 **에러 트레이스백 전문**을 모델에 줘 수정안을 받습니다. 핵심은 **요약하지 말고 전문(全文)**.

⌨️ **터미널 Codex:** `codex "이 실패한 테스트의 트레이스백과 코드를 보고 원인과 수정안을 알려줘"`

In [6]:
import traceback

# 버그 주입: 예산 필터를 빼버린 버전 (조용한 회귀)
def within_budget_buggy(restaurants, budget):
    if budget < 0:
        raise ValueError("budget은 0 이상이어야 합니다")
    return sorted(restaurants, key=lambda r: r["price"])   # ← budget 필터 누락 (버그)

print("=== 버그 버전으로 같은 테스트 실행 ===")
run_tests(generated, {"within_budget": within_budget_buggy})
# 일부 테스트가 ❌ FAIL 이면, 생성된 테스트가 회귀를 제대로 잡는다는 뜻!

# 실패한 테스트 1건의 에러 전문을 수집
err_text = "(에러 없음)"
ns = dict({"within_budget": within_budget_buggy})
exec(generated, ns)
for name, fn in {k: v for k, v in ns.items() if k.startswith("test_") and callable(v)}.items():
    try:
        fn()
    except Exception:
        err_text = traceback.format_exc()
        break
print("\n수집한 에러 전문:\n", err_text)

DEBUG_PROMPT = f"""다음 파이썬 함수가 테스트를 통과하지 못한다. 원인을 한 줄로 설명하고, 수정한 전체 코드를 제시하라.

[코드]
{inspect.getsource(within_budget_buggy)}

[실패한 테스트 트레이스백 전문]
{err_text}
"""

fix = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": DEBUG_PROMPT}],
    temperature=0,
).choices[0].message.content
print("\n🔧 모델의 진단·수정안:\n", fix)

=== 버그 버전으로 같은 테스트 실행 ===
✅ PASS · test_within_budget_normal_case
✅ PASS · test_within_budget_boundary_case
❌ FAIL · test_within_budget_empty_result → AssertionError: 
✅ PASS · test_within_budget_negative_budget

결과: 3 passed, 1 failed (4 tests)

수집한 에러 전문:
 Traceback (most recent call last):
  File "/var/folders/g3/6ywsl8nj20v0z3c9_c1y320r0000gn/T/ipykernel_15359/1084059470.py", line 19, in <module>
    fn()
  File "<string>", line 34, in test_within_budget_empty_result
AssertionError


🔧 모델의 진단·수정안:
 원인은 함수가 예산(budget) 내의 레스토랑만 필터링하지 않고 모든 레스토랑을 정렬하여 반환하기 때문입니다. 

수정한 전체 코드는 다음과 같습니다:

```python
def within_budget(restaurants, budget):
    if budget < 0:
        raise ValueError("budget은 0 이상이어야 합니다")
    # 예산 내의 레스토랑만 필터링
    within_budget_restaurants = [r for r in restaurants if r["price"] <= budget]
    return sorted(within_budget_restaurants, key=lambda r: r["price"])
```

이제 이 코드는 주어진 예산 내의 레스토랑만 필터링하여 정렬된 리스트를 반환합니다.


## STEP 6 — pass@k: LLM 출력의 흔들림을 잡는다

결정적 함수는 몇 번을 돌려도 같지만, LLM 출력은 흔들립니다. 같은 검증을 k번 돌려 **pass@k**로 안정성을 잽니다.

In [7]:
K = 3
question = "강남에서 2만원 이하 매운 점심 추천해줘"
passes = 0
for i in range(K):
    r = client.chat.completions.create(model=MODEL, temperature=1.0,
        messages=[{"role":"user","content":question + " 반드시 가격(원)을 포함해 답하라."}])
    out = r.choices[0].message.content
    ok = "원" in out                      # 결정적 체크: 가격 표기가 있는가
    passes += ok
    print(f"  시도 {i+1}: {'✅' if ok else '❌'}  {out[:60]}...")
print(f"pass@{K} = {passes}/{K} — 3/3이 아니면 프롬프트로 형식을 더 못박아야 한다")

  시도 1: ✅  강남에서 2만원 이하의 매운 점심으로 추천할 만한 몇 가지 메뉴를 소개할게요.

1. **매운 닭개장 (닭고...
  시도 2: ✅  강남에서 2만원 이하의 매운 점심 메뉴로 다음을 추천합니다:

1. **몇몇 매운 떡볶이** - 7,000원...
  시도 3: ✅  강남에서 2만원 이하의 매운 점심으로 다음과 같은 식당과 메뉴를 추천합니다:

1. **청국장&된장 비빔밥*...
pass@3 = 3/3 — 3/3이 아니면 프롬프트로 형식을 더 못박아야 한다


## STEP 7 — 조용한 실패 감지: 지연 스파이크 (p95)

강의의 '조용한 실패 4종' 중 지연 — 평균은 멀쩡해도 꼬리(p95)가 폭발할 수 있습니다.

In [8]:
import time, statistics
lat = []
for i in range(5):
    t0 = time.time()
    client.chat.completions.create(model=MODEL,
        messages=[{"role":"user","content":f"숫자 {i}를 한국어로"}])
    lat.append((time.time()-t0)*1000)
lat_sorted = sorted(lat)
p95 = lat_sorted[int(len(lat_sorted)*0.95)-1]
print(f"지연(ms): {[round(x) for x in lat]}")
print(f"평균 {statistics.mean(lat):.0f}ms · p95 {p95:.0f}ms")
LIMIT = 5000
print("⚠ p95 스파이크 — CI에서 성능 테스트로 잡아야 할 신호" if p95 > LIMIT else "✅ p95 임계 이내")

지연(ms): [796, 1082, 828, 2203, 812]
평균 1144ms · p95 1082ms
✅ p95 임계 이내


## STEP 8 — ⭐ 내 MVP에 적용

위 흐름(서비스 정의 → 테스트 자동 생성·실행 → 스냅샷 회귀 → AI 디버깅)은 **그대로** 당신 팀 MVP에 옮길 수 있습니다.
`my_service`를 **우리 서비스의 핵심 기능**으로 바꾸면 끝 — 결정적 로직은 단위 테스트로, LLM 출력은 스냅샷으로.

⌨️ **터미널 Codex:** `codex "우리 MVP 핵심 함수의 정상·경계·예외 테스트를 만들고 실행한 뒤, 출력 스냅샷 회귀까지 걸어줘"`

In [9]:
# 팀별 작성: 우리 MVP의 '핵심 기능' 1개를 적어보세요 (이름/설명/결정적 로직 vs LLM 출력)
MY_SERVICE_TODO = {
  # "name": "add_expense",                       # 예: 가계부 MVP
  # "deterministic": "금액 검증·합계 계산 (단위 테스트 대상)",
  # "llm_output": "지출 한 줄 코멘트 생성 (스냅샷 대상)",
}
print("우리 MVP 테스트 대상:", MY_SERVICE_TODO or "⬜ 채워주세요")
print("→ 채운 뒤 STEP 1의 within_budget/my_service 자리에 우리 함수를 넣고")
print("  STEP 2~5(생성·실행·스냅샷·디버깅)를 그대로 다시 돌리면 '내 서비스 자동 테스트' 완성")

우리 MVP 테스트 대상: ⬜ 채워주세요
→ 채운 뒤 STEP 1의 within_budget/my_service 자리에 우리 함수를 넣고
  STEP 2~5(생성·실행·스냅샷·디버깅)를 그대로 다시 돌리면 '내 서비스 자동 테스트' 완성


## 정리

- **생성(테스트 위임) + 회귀(스냅샷 diff) + 디버깅(에러 전문) + CI(자동 실행)** = AI 시대의 테스트 루프.
- AI가 빨리 짤수록 **자동 검증이 더 중요**해진다 — 테스트는 코드를, **사람은 테스트를** 신뢰한다.
- LLM 출력은 **비결정적** → exact-match 대신 **스냅샷 diff**로 변화를 본다. diff는 "버그"가 아니라 **"변화 알림"**.